# Лабораторна робота 5
## Очищення даних, інженерія ознак та зменшення розмірності

**Предметна область:** нерухомість · машинне навчання · класифікація (продовження ЛР 4).
**Задача:** класифікація об'єкта на три цінові сегменти (низький / середній / високий) за характеристиками.
**Метрика:** macro-F1. **Схема перевірки:** `TimeSeriesSplit` (дані впорядковані за датою).

**Дані:** `Melbourne_housing_FULL.csv` (Kaggle, ~34 тис. записів; є пропуски, категорії, числові ознаки, дати, дублікати).
Набір Ames Housing з ЛР 4 має лише 2 930 записів, тому вимогу «≥ 5000 записів» він не виконує. Якщо хочете лишитися на іншому наборі, змініть `DATA_PATH` та списки `NUM_BASE`, `CAT` у комірці налаштувань.

> Шлях до файлу можна задати змінною середовища `LAB5_DATA`. Для швидкого прогону використайте `LAB5_QUICK=1`.

In [ ]:
import os, time, warnings, re, difflib, platform
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import sklearn
from scipy import stats
from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import TimeSeriesSplit, GroupKFold, StratifiedKFold, cross_val_score
from sklearn.metrics import f1_score, classification_report

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
QUICK = os.environ.get("LAB5_QUICK") == "1"          # швидкий прогін для перевірки
DATA_PATH = Path(os.environ.get("LAB5_DATA", "Melbourne_housing_FULL.csv"))
N_TREES = 40 if QUICK else 150
Path("figures").mkdir(exist_ok=True)
plt.rcParams["figure.dpi"] = 110

def savefig(name):
    plt.savefig(f"figures/{name}.png", bbox_inches="tight")

# Налаштування набору даних
NUM_BASE = ["Rooms","Distance","Bedroom2","Bathroom","Car","Landsize","BuildingArea",
            "YearBuilt","Lattitude","Longtitude","Propertycount"]
CAT = ["Type","Method","Regionname","CouncilArea"]
NEW = ["Month_sin","Month_cos","HouseAge","Build_to_Land","Rooms_per_Bath","Area_per_Room"]
print("sklearn", sklearn.__version__, "| pandas", pd.__version__, "| QUICK =", QUICK)

## 0. Завантаження даних

Рядки без цінових міток (`Price`) і без дати видаляємо: це **ціль**, а не ознака, тому її не можна імпутувати. Далі дані сортуються за часом.

In [ ]:
df = pd.read_csv(DATA_PATH)
df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")
for c in NUM_BASE + ["Price"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
for c in CAT + ["Suburb", "Address"]:          # єдиний тип object із np.nan для пропусків
    df[c] = df[c].astype(object).where(df[c].notna(), np.nan)
n_raw = len(df)
df = df.dropna(subset=["Price", "Date"]).reset_index(drop=True)
print(f"Було {n_raw} записів, після видалення рядків без Price/Date: {len(df)}")
if len(df) < 5000:
    print("УВАГА: за умовою лабораторної потрібно ≥ 5000 записів")
df.head()

## Результат 1. Очищення: пропуски й дублікати

### Крок 1. Механізм пропусків
Для ознак `Car`, `BuildingArea`, `YearBuilt` порівнюємо розподіл інших ознак у групах «є пропуск» / «немає пропуску» (тест Манна–Вітні). Якщо розподіли значущо різні, пропуск залежить від спостережуваних ознак, тобто це **MAR**, а не MCAR. MNAR (залежність від самого пропущеного значення) за даними перевірити неможливо, її можна лише обґрунтувати змістовно.

In [ ]:
MISS_COLS = ["Car", "BuildingArea", "YearBuilt"]
miss_rate = df[NUM_BASE + CAT].isna().mean().sort_values(ascending=False)
print("Частка пропусків:\n", miss_rate.round(3).to_string())

rows = []
for c in MISS_COLS:
    flag = df[c].isna()
    for o in [x for x in NUM_BASE if x != c]:
        a, b = df.loc[flag, o].dropna(), df.loc[~flag, o].dropna()
        if len(a) > 20 and len(b) > 20:
            rows.append((c, o, a.median(), b.median(), stats.mannwhitneyu(a, b).pvalue))
miss_tbl = pd.DataFrame(rows, columns=["ознака з пропусками", "інша ознака",
                                       "медіана (є пропуск)", "медіана (немає)", "p-value"])
display(miss_tbl.round(4).sort_values("p-value").head(12))

sig = miss_tbl.assign(sig=miss_tbl["p-value"] < 0.001).groupby("ознака з пропусками")["sig"].sum()
print("\nК-сть ознак із значущою різницею (p<0.001):\n", sig.to_string())
for c, k in sig.items():
    print(f"{c}: ", "ймовірно MAR (пропуск пов'язаний з іншими ознаками)" if k >= 2 else "схоже на MCAR")

fig, ax = plt.subplots(figsize=(6, 3))
miss_rate[miss_rate > 0].plot.bar(ax=ax, color="#4c72b0")
ax.set_ylabel("частка пропусків"); ax.set_title("Пропуски за ознаками")
savefig("r1_missing"); plt.show()

**Висновок щодо механізму (відредагуйте під свої результати):** якщо у виводі вище більшість порівнянь значущі, то пропуски в `BuildingArea` / `YearBuilt` залежать від типу й розташування об'єкта (MAR). Наприклад, для квартир площа будівлі та рік часто не вказуються. Тому доцільно додавати **індикатор пропуску** й використовувати імпутацію за сусідніми об'єктами (KNN).

### Крок 4. Дублікати (точні й нечіткі)
Точні дублікати: збіг усіх полів. Нечіткі: однакові `Suburb` + `Date` + `Rooms` і схожі адреси (`difflib`, коефіцієнт ≥ 0.9). Такі записи **об'єднуються** (`groupby().first()` бере перше непорожнє значення кожної ознаки).

In [ ]:
def norm_addr(s):
    return re.sub(r"[^a-z0-9 ]", "", str(s).lower()).strip()

n0 = len(df)
n_exact = int(df.drop(columns=[]).duplicated().sum())
tmp = df.copy()
tmp["_addr"] = tmp["Address"].map(norm_addr)
rep = np.arange(len(tmp))
for _, idx in tmp.groupby(["Suburb", "Date"], sort=False).indices.items():
    if len(idx) < 2 or len(idx) > 200:
        continue
    for a in range(len(idx)):
        i = idx[a]
        if rep[i] != i:
            continue
        for j in idx[a + 1:]:
            if (rep[j] == j and tmp.at[i, "Rooms"] == tmp.at[j, "Rooms"] and
                    difflib.SequenceMatcher(None, tmp.at[i, "_addr"], tmp.at[j, "_addr"]).ratio() >= 0.9):
                rep[j] = i
merged = (tmp.assign(_rep=rep).groupby("_rep", sort=False).first()
             .drop(columns="_addr").reset_index(drop=True))
n_total = n0 - len(merged)
print(f"Точних дублікатів: {n_exact}")
print(f"Нечітких (схожа адреса, та сама дата/район/кімнати): {n_total - n_exact}")
print(f"Усього видалено/об'єднано: {n_total}; залишилось {len(merged)} із {n0}")
df = merged
for c in NUM_BASE + ["Price"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")
for c in CAT + ["Suburb", "Address"]:
    df[c] = df[c].astype(object).where(df[c].notna(), np.nan)

### Розбиття на train / test і формування цілі
Дані сортуємо за датою; **перші 85 % — train**, останні 15 % — фінальний тест (майбутнє відносно train). Порогові значення цінових сегментів (терцилі) обчислюємо **лише по train**, щоб не було витоку.

In [ ]:
df = df.sort_values("Date", kind="stable").reset_index(drop=True)
n_train = int(0.85 * len(df))
q1_, q2_ = df.loc[:n_train - 1, "Price"].quantile([1/3, 2/3])
y = pd.cut(df["Price"], [-np.inf, q1_, q2_, np.inf], labels=[0, 1, 2]).astype(int)
X = df.drop(columns=["Price"])
X_train, y_train = X.iloc[:n_train], y.iloc[:n_train]
X_test,  y_test  = X.iloc[n_train:], y.iloc[n_train:]
tscv = TimeSeriesSplit(n_splits=5)
CLASS_NAMES = ["низький", "середній", "високий"]
print(f"train={len(X_train)} ({X_train.Date.min().date()}–{X_train.Date.max().date()}), "
      f"test={len(X_test)} ({X_test.Date.min().date()}–{X_test.Date.max().date()})")
print(f"Пороги сегментів: {q1_:,.0f} та {q2_:,.0f}")
print(y_train.value_counts(normalize=True).sort_index().round(3).to_string())

## Спільні компоненти: трансформери, `build_pipeline`, `evaluate`
Усе, що «навчається» на даних (межі winsorizing, медіани, масштаб, PCA), живе всередині `Pipeline` і обчислюється **тільки на train-фолді**. Це і є захист від витоку.

In [ ]:
class FeatureAdder(BaseEstimator, TransformerMixin):
    """Нові ознаки: ознаки з дати (sin/cos місяця), вік будинку, співвідношення площ."""
    def __init__(self, add=True):
        self.add = add
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        X = X.copy()
        if not self.add:
            return X
        d = pd.to_datetime(X["Date"])
        m = d.dt.month
        X["Month_sin"] = np.sin(2 * np.pi * m / 12)
        X["Month_cos"] = np.cos(2 * np.pi * m / 12)
        X["HouseAge"] = (d.dt.year - X["YearBuilt"]).clip(lower=0)
        X["Build_to_Land"] = X["BuildingArea"] / X["Landsize"].replace(0, np.nan)
        X["Rooms_per_Bath"] = X["Rooms"] / X["Bathroom"].replace(0, np.nan)
        X["Area_per_Room"] = X["BuildingArea"] / X["Rooms"].replace(0, np.nan)
        return X

OUT_COLS = ["Rooms","Distance","Bedroom2","Bathroom","Car","Landsize","BuildingArea","YearBuilt",
            "Propertycount","HouseAge","Build_to_Land","Rooms_per_Bath","Area_per_Room"]
LOG_COLS = ["Landsize","BuildingArea","Build_to_Land","Area_per_Room"]

class OutlierHandler(BaseEstimator, TransformerMixin):
    """strategy: 'none' | 'winsor' (обрізання за 1%/99% train) | 'winsor_log' (обрізання + log1p)."""
    def __init__(self, strategy="none", q=(0.01, 0.99)):
        self.strategy = strategy
        self.q = q
    def fit(self, X, y=None):
        self.cols_ = [c for c in OUT_COLS if c in X.columns]
        Z = X[self.cols_].astype(float)
        self.lo_, self.hi_ = Z.quantile(self.q[0]), Z.quantile(self.q[1])
        return self
    def transform(self, X):
        X = X.copy()
        if self.strategy in ("winsor", "winsor_log"):
            X[self.cols_] = X[self.cols_].astype(float).clip(self.lo_, self.hi_, axis=1)
        if self.strategy == "winsor_log":
            for c in LOG_COLS:
                if c in X.columns:
                    X[c] = np.log1p(X[c].astype(float).clip(lower=0))
        return X

def make_model(name):
    if name == "rf":
        return RandomForestClassifier(n_estimators=N_TREES, min_samples_leaf=2, n_jobs=-1, random_state=SEED)
    if name == "lr":
        return LogisticRegression(max_iter=500, random_state=SEED)
    raise ValueError(name)

def build_pipeline(impute="median", outliers="none", features=False, scale=True, pca=None, model="rf"):
    """impute: 'constant' (технічне -1) | 'median' (+індикатор) | 'knn' (+індикатор)."""
    num_cols = NUM_BASE + (NEW if features else [])
    steps_num = []
    if scale:
        steps_num.append(("scaler", StandardScaler()))
    if impute == "median":
        steps_num.append(("imputer", SimpleImputer(strategy="median", add_indicator=True)))
    elif impute == "knn":
        steps_num.append(("imputer", KNNImputer(n_neighbors=5, add_indicator=True)))
    else:
        steps_num.append(("imputer", SimpleImputer(strategy="constant", fill_value=-1)))
    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False, min_frequency=10)),
    ])
    ct = ColumnTransformer([("num", Pipeline(steps_num), num_cols), ("cat", cat_pipe, CAT)])
    steps = [("feat", FeatureAdder(features)), ("out", OutlierHandler(outliers)), ("prep", ct)]
    if pca:
        steps.append(("pca", PCA(n_components=pca, random_state=SEED)))
    steps.append(("clf", make_model(model)))
    return Pipeline(steps)

def evaluate(pipe, X, y, cv=None, groups=None, row_filter=None):
    """CV macro-F1 (середнє, σ) + середній час навчання. row_filter застосовується ЛИШЕ до train-фолда."""
    cv = cv or TimeSeriesSplit(n_splits=5)
    scores, times = [], []
    for tr, va in cv.split(X, y, groups):
        Xtr, ytr = X.iloc[tr], y.iloc[tr]
        if row_filter is not None:
            mask = row_filter(Xtr)
            Xtr, ytr = Xtr[mask], ytr[mask]
        p = clone(pipe)
        t = time.time(); p.fit(Xtr, ytr); times.append(time.time() - t)
        scores.append(f1_score(y.iloc[va], p.predict(X.iloc[va]), average="macro"))
    return dict(mean=float(np.mean(scores)), std=float(np.std(scores, ddof=1)),
                fit_time=float(np.mean(times)), folds=scores)

def fmt(r):
    return f"{r['mean']:.4f} ± {r['std']:.4f}"

### Кроки 2–3. Стратегії імпутації в `Pipeline` + порівняння на крос-валідації
Порівнюємо: (0) технічне заповнення `-1` як базова лінія, (1) **медіана + індикатор пропуску**, (2) **KNNImputer**. Імпутер стоїть усередині `Pipeline`, тому медіани та сусіди визначаються тільки по train-фолду. Оцінка через `cross_val_score`.

In [ ]:
imp_variants = [("constant −1 (базова)", "constant"),
                ("медіана + індикатор", "median"),
                ("KNNImputer (k=5) + індикатор", "knn")]
imp_rows = []
for name, imp in imp_variants:
    pipe = build_pipeline(impute=imp, scale=True, model="rf")
    t = time.time()
    s = cross_val_score(pipe, X_train, y_train, cv=tscv, scoring="f1_macro")
    imp_rows.append(dict(стратегія=name, F1_mean=s.mean(), F1_std=s.std(ddof=1), час_с=time.time() - t))
imp_df = pd.DataFrame(imp_rows)
display(imp_df.round(4))
BEST_IMPUTE = dict(imp_variants)[imp_df.loc[imp_df.F1_mean.idxmax(), "стратегія"]]
print("Найкраща імпутація за CV:", BEST_IMPUTE)

### Крок 5. Обґрунтування рішень (по одному реченню)
Відредагуйте після власного запуску.
- **Рядки без `Price` видалено**, бо ціль не можна імпутувати без «підгонки» відповіді.
- **Індикатор пропуску додано**, бо пропуски MAR і саме факт відсутності (напр. немає площі будівлі) інформативний для ціни.
- **Імпутацію винесено в `Pipeline`**, щоб медіани/сусіди рахувались тільки по train-фолду (без витоку).
- **Точні та нечіткі дублікати об'єднано**, бо той самий об'єкт у train і test завищує метрику.

## Результат 2. Викиди

### Кроки 1–2. Три способи позначити викиди та їх перетин
Аналіз виконується **тільки на train**. Рядок вважається викидом, якщо хоч одна з числових ознак позначена (IQR 1.5·IQR; |z| > 3) або IsolationForest (contamination = 2 %) виніс його як аномалію.

In [ ]:
OUT_DETECT = ["Rooms","Distance","Bathroom","Car","Landsize","BuildingArea","YearBuilt","Propertycount"]
Xo = X_train[OUT_DETECT].astype(float)
q1, q3 = Xo.quantile(.25), Xo.quantile(.75); iqr = q3 - q1
f_iqr = (Xo < q1 - 1.5 * iqr) | (Xo > q3 + 1.5 * iqr)
f_z = ((Xo - Xo.mean()) / Xo.std()).abs() > 3
Xf = (Xo.fillna(Xo.median()) - Xo.mean()) / Xo.std()
iso = IsolationForest(contamination=0.02, random_state=SEED, n_jobs=-1).fit(Xf)
f_iso = pd.Series(iso.predict(Xf) == -1, index=Xo.index)
flags = pd.DataFrame({"IQR": f_iqr.any(axis=1), "z>3": f_z.any(axis=1), "IsolationForest": f_iso})

cnt = pd.DataFrame({"к-сть рядків": flags.sum(), "частка, %": (flags.mean() * 100).round(2)})
display(cnt)
print("Позначення за ознаками (IQR / z):")
display(pd.DataFrame({"IQR": f_iqr.sum(), "z>3": f_z.sum()}))

combos = flags.groupby(list(flags.columns)).size().rename("к-сть").reset_index()
combos["комбінація"] = combos.apply(lambda r: "+".join([c for c in flags.columns if r[c]]) or "жоден", axis=1)
combos = combos.sort_values("к-сть", ascending=False)
display(combos[["комбінація", "к-сть"]].reset_index(drop=True))

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
cnt["к-сть рядків"].plot.bar(ax=ax[0], color="#55a868"); ax[0].set_title("Кількість викидів за методом")
cc = combos[combos["комбінація"] != "жоден"]
ax[1].barh(cc["комбінація"], cc["к-сть"], color="#c44e52"); ax[1].set_title("Перетин методів"); ax[1].invert_yaxis()
plt.tight_layout(); savefig("r2_outliers"); plt.show()

### Крок 3. Аналіз 10 прикладів
Беремо випадкові рядки, позначені ≥ 2 методами. Колонка «гіпотеза» формується простими правилами, це **підказка**. Висновок «помилка / рідкісна подія / реальне значення» студент має підтвердити сам, переглянувши рядок.

In [ ]:
def hint(r):
    if pd.notna(r.YearBuilt) and (r.YearBuilt < 1800 or r.YearBuilt > 2026):
        return "помилка (неможливий рік побудови)"
    if pd.notna(r.BuildingArea) and r.BuildingArea == 0:
        return "помилка (нульова площа)"
    if pd.notna(r.BuildingArea) and pd.notna(r.Landsize) and r.Landsize > 0 and r.BuildingArea > 3 * r.Landsize:
        return "імовірна помилка (площа будівлі ≫ ділянки)"
    if pd.notna(r.Landsize) and r.Landsize > 20000:
        return "рідкісна подія (дуже велика ділянка)"
    if pd.notna(r.Rooms) and r.Rooms >= 8:
        return "рідкісна подія (великий об'єкт)"
    if pd.notna(r.Distance) and r.Distance > 40:
        return "реальне значення (віддалений район)"
    return "реальне значення / потребує перевірки"

cand = flags.index[flags.sum(axis=1) >= 2]
if len(cand) < 10:
    cand = flags.index[flags.sum(axis=1) >= 1]
pick = pd.Series(cand).sample(min(10, len(cand)), random_state=SEED)
ex = X_train.loc[pick, ["Suburb","Type","Rooms","Distance","Bathroom","Car","Landsize","BuildingArea","YearBuilt"]].copy()
ex["Price"] = df.loc[pick, "Price"]
ex["гіпотеза"] = ex.apply(hint, axis=1)
display(ex)
ex["гіпотеза"].value_counts()

### Кроки 4–5. Вибір стратегії (лише train) та вплив на метрику
Порівнюємо чотири варіанти: **залишити**, **winsorizing 1 %/99 %**, **winsorizing + log1p** (для асиметричних площ), **видалити екстремуми з train-фолду** (3·IQR; тестова частина не змінюється). Перші три живуть у `Pipeline` і навчаються по train-фолду.

In [ ]:
def drop_extreme(Xtr):
    Z = Xtr[OUT_DETECT].astype(float)
    a, b = Z.quantile(.25), Z.quantile(.75); r = b - a
    bad = ((Z < a - 3 * r) | (Z > b + 3 * r)).any(axis=1)
    return ~bad.values

out_rows, out_res = [], {}
for name, strat, rf_ in [("залишити", "none", None),
                         ("winsorizing 1%/99%", "winsor", None),
                         ("winsorizing + log1p", "winsor_log", None),
                         ("видалити з train (3·IQR)", "none", drop_extreme)]:
    r = evaluate(build_pipeline(impute=BEST_IMPUTE, outliers=strat, scale=True, model="rf"),
                 X_train, y_train, row_filter=rf_)
    out_res[name] = (strat, r)
    out_rows.append(dict(стратегія=name, F1=fmt(r), F1_mean=r["mean"]))
out_df = pd.DataFrame(out_rows)
display(out_df[["стратегія", "F1"]])
pipeline_ok = {k: v for k, v in out_res.items() if not k.startswith("видалити")}
best_name = max(pipeline_ok, key=lambda k: pipeline_ok[k][1]["mean"])
BEST_OUT = pipeline_ok[best_name][0]
ABL_OUT = BEST_OUT if BEST_OUT != "none" else "winsor"
print("Обрана стратегія (серед pipeline-сумісних):", best_name, "→", BEST_OUT)

**Висновок (відредагуйте):** порівняйте значення в таблиці. Для дерев (RandomForest) викиди майже не шкодять, бо дерева інваріантні до монотонних перетворень. Тому ефект може бути в межах σ. Якщо різниця менша за σ, чесний висновок: «викиди суттєво не впливають на RF; winsorizing лишаємо для стабільності лінійної моделі».

## Результат 3. Ознаки та Pipeline без витоку

**Нові ознаки (6):** `Month_sin`, `Month_cos` (циклічність місяця продажу), `HouseAge` (рік продажу − рік побудови), `Build_to_Land` (площа будівлі / ділянки), `Rooms_per_Bath`, `Area_per_Room`.
Усі вони обчислюються **рядок за рядком** (без агрегатів по всьому набору), тому самі не створюють витоку.

**Кодування й масштаб:** `OneHotEncoder(handle_unknown="ignore")` для категорій, `StandardScaler` для числових; все в `ColumnTransformer` + `Pipeline` разом із моделлю.

### Схема розбиття
Оголошення впорядковані за датою, а ринок має тренд, тому **випадкове розбиття дозволяє «заглядати в майбутнє»**: навчальна вибірка містить сусідні за часом об'єкти (та їхні ціни), які фактично відомі лише після тестового. `TimeSeriesSplit` завжди тренує на минулому й перевіряє на майбутньому. Для порівняння наведено `GroupKFold` за районом (перевірка узагальнення на нові райони) та випадковий `StratifiedKFold` як демонстрацію завищеної оцінки.

In [ ]:
full_pipe = build_pipeline(impute=BEST_IMPUTE, outliers=ABL_OUT, features=True, scale=True, model="rf")
display(full_pipe)

schemes = [("TimeSeriesSplit(5) — основна", dict(cv=TimeSeriesSplit(n_splits=5))),
           ("GroupKFold(5) за Suburb", dict(cv=GroupKFold(n_splits=5), groups=X_train["Suburb"].fillna("NA"))),
           ("StratifiedKFold(5, shuffle) — для порівняння", dict(cv=StratifiedKFold(5, shuffle=True, random_state=SEED)))]
r3 = []
for name, kw in schemes:
    r = evaluate(full_pipe, X_train, y_train, **kw)
    r3.append(dict(схема=name, **{"macro-F1 ± σ": fmt(r)}))
no_feat = evaluate(build_pipeline(impute=BEST_IMPUTE, outliers=ABL_OUT, features=False, scale=True, model="rf"),
                   X_train, y_train)
r3.append(dict(схема="TimeSeriesSplit, без нових ознак", **{"macro-F1 ± σ": fmt(no_feat)}))
display(pd.DataFrame(r3))

full_pipe.fit(X_train, y_train)
pred = full_pipe.predict(X_test)
print(f"Фінальний тест (останні 15 % за часом): macro-F1 = {f1_score(y_test, pred, average='macro'):.4f}")
print(classification_report(y_test, pred, target_names=CLASS_NAMES, digits=3))

**Висновок (відредагуйте):** `TimeSeriesSplit` дає чесну (зазвичай нижчу) оцінку, ніж випадкове розбиття. Різниця між ними і є «вартість» витоку. Метрику для звіту беріть з першого рядка таблиці (± σ за 5 фолдами).

## Результат 4. Зменшення розмірності

### Кроки 1–2. PCA: scree, накопичена дисперсія, k за 95 %
Числові ознаки (включно з новими) → winsorizing (за вибраною стратегією) → стандартизація → медіанна імпутація → PCA. Усе навчається на train.

In [ ]:
num_cols_all = NUM_BASE + NEW
prep_num = Pipeline([
    ("feat", FeatureAdder(True)),
    ("out", OutlierHandler(ABL_OUT)),
    ("sel", ColumnTransformer([("n", Pipeline([("sc", StandardScaler()),
                                               ("imp", SimpleImputer(strategy="median"))]), num_cols_all)])),
])
Z = prep_num.fit_transform(X_train)
pca_full = PCA(random_state=SEED).fit(Z)
evr = pca_full.explained_variance_ratio_
cum = np.cumsum(evr)
K95 = int(np.argmax(cum >= 0.95) + 1)
print(f"Числових ознак: {Z.shape[1]}; k для 95 % дисперсії = {K95}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
ax[0].plot(range(1, len(evr) + 1), evr, "o-"); ax[0].set_title("Scree plot")
ax[0].set_xlabel("компонента"); ax[0].set_ylabel("частка дисперсії")
ax[1].plot(range(1, len(cum) + 1), cum, "o-"); ax[1].axhline(0.95, color="r", ls="--")
ax[1].axvline(K95, color="gray", ls=":"); ax[1].set_title(f"Накопичена дисперсія (k95 = {K95})")
ax[1].set_xlabel("кількість компонент")
plt.tight_layout(); savefig("r4_pca_variance"); plt.show()

load = pd.DataFrame(pca_full.components_[:2].T, index=num_cols_all, columns=["PC1", "PC2"])
for pc in ["PC1", "PC2"]:
    print(f"\nНайбільші навантаження в {pc}:")
    print(load[pc].reindex(load[pc].abs().sort_values(ascending=False).index).head(5).round(3).to_string())

### Крок 3. t-SNE / UMAP (зміна `perplexity` / `n_neighbors`), розфарбовування за класом
Для швидкості береться випадкова підвибірка train.

In [ ]:
SUB = 1500 if QUICK else 4000
rng = np.random.RandomState(SEED)
idx = rng.choice(len(Z), size=min(SUB, len(Z)), replace=False)
Zs, ys = Z[idx], y_train.values[idx]
colors = ["#4c72b0", "#dd8452", "#c44e52"]

def scatter(ax, E, title):
    for k in range(3):
        ax.scatter(E[ys == k, 0], E[ys == k, 1], s=4, c=colors[k], label=CLASS_NAMES[k], alpha=.7)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])

fig, ax = plt.subplots(1, 3, figsize=(14, 4))
for a, p in zip(ax, [5, 30, 50]):
    scatter(a, TSNE(n_components=2, perplexity=p, init="pca", random_state=SEED).fit_transform(Zs), f"t-SNE, perplexity={p}")
ax[0].legend(markerscale=3); plt.tight_layout(); savefig("r4_tsne"); plt.show()

try:
    import umap
    fig, ax = plt.subplots(1, 3, figsize=(14, 4))
    for a, nn in zip(ax, [5, 15, 50]):
        scatter(a, umap.UMAP(n_neighbors=nn, min_dist=0.1, random_state=SEED).fit_transform(Zs), f"UMAP, n_neighbors={nn}")
    ax[0].legend(markerscale=3); plt.tight_layout(); savefig("r4_umap"); plt.show()
except ImportError:
    print("umap-learn не встановлено (pip install umap-learn); t-SNE достатньо за умовою «t-SNE і/або UMAP».")

### Крок 4. Якість моделі: повні ознаки vs PCA (95 %): метрика та час навчання

In [ ]:
rows = []
for m in ["rf", "lr"]:
    for label, pc in [("повні ознаки", None), ("PCA 95 %", 0.95)]:
        pipe = build_pipeline(impute=BEST_IMPUTE, outliers=ABL_OUT, features=True, scale=True, pca=pc, model=m)
        r = evaluate(pipe, X_train, y_train)
        ncomp = pipe.fit(X_train, y_train)[:-1].transform(X_train.iloc[:5]).shape[1]
        rows.append(dict(модель={"rf": "RandomForest", "lr": "LogisticRegression"}[m], конфігурація=label,
                         ознак_на_вході_моделі=ncomp, **{"macro-F1 ± σ": fmt(r)}, час_навчання_с=round(r["fit_time"], 2)))
display(pd.DataFrame(rows))

### Крок 5. Обмеження t-SNE / UMAP
- Осі та їхні одиниці **не мають змісту**: це не головні компоненти й не ознаки.
- **Відстані між кластерами** і розміри кластерів не інтерпретуються кількісно: t-SNE зберігає лише локальні сусідства, глобальна геометрія спотворюється (результат залежить від `perplexity`, UMAP від `n_neighbors` і `min_dist`).
- Проєкція **стохастична** (залежить від seed) і не має оберненого перетворення; нову точку можна проєктувати лише приблизно (для UMAP є `transform`, для t-SNE немає).
- Видимі «острівці» можуть бути артефактом параметрів, тому їх треба перевіряти на кількох значеннях параметрів, як зроблено вище.
- Це інструмент для **візуального аналізу**, а не для прийняття рішень чи ознак для моделі без додаткової перевірки.

## Результат 5. Абляція: вплив кроків підготовки

- **Базова («сирі дані»):** технічне заповнення `-1` для числових, `'missing'` для категорій + OneHot, без викидів, без нових ознак, без масштабування, без PCA (інакше модель не запуститься).
- **Повна («усі кроки»):** імпутація (медіана + індикатор), викиди (обрана стратегія), нові ознаки, масштабування, PCA 95 %.
- Експерименти: кумулятивне додавання кроків, «лише один крок» до сирих даних, «усі крім одного». Однакове розбиття (`TimeSeriesSplit(5)`) і seed для всіх. Дві моделі: `rf` (нечутлива до масштабу) і `lr` (чутлива). Так видно, для кого який крок корисний.

In [ ]:
ABL_MODELS = ["rf", "lr"]
STEPS = ["impute", "outliers", "features", "scale", "pca"]
LABEL = {"impute": "Імпутація", "outliers": "Викиди", "features": "Нові ознаки", "scale": "Масштабування", "pca": "PCA"}

def cfg(active, model):
    return dict(impute="median" if "impute" in active else "constant",
                outliers=ABL_OUT if "outliers" in active else "none",
                features="features" in active, scale="scale" in active,
                pca=0.95 if "pca" in active else None, model=model)

experiments = [("Сирі дані", "raw", None, frozenset())]
for i, s in enumerate(STEPS):
    experiments.append((f"Кумулятивно до «{LABEL[s]}»", "cum", s, frozenset(STEPS[:i + 1])))
for s in STEPS:
    experiments.append((f"Лише «{LABEL[s]}»", "add", s, frozenset([s])))
for s in STEPS:
    experiments.append((f"Усі, крім «{LABEL[s]}»", "drop", s, frozenset(STEPS) - {s}))

cache, abl_rows = {}, []
for name, kind, step, active in experiments:
    for m in ABL_MODELS:
        key = (active, m)
        if key not in cache:
            cache[key] = evaluate(build_pipeline(**cfg(active, m)), X_train, y_train)
        r = cache[key]
        abl_rows.append(dict(експеримент=name, тип=kind, крок=step, модель=m, mean=r["mean"], std=r["std"]))
abl = pd.DataFrame(abl_rows)
tbl = abl.assign(v=abl.apply(lambda r: f"{r['mean']:.4f} ± {r['std']:.4f}", axis=1)) \
         .pivot(index="експеримент", columns="модель", values="v")
tbl = tbl.loc[[e[0] for e in experiments]]
display(tbl)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
# (а) кумулятивний графік приросту
for m, col in zip(ABL_MODELS, ["#4c72b0", "#c44e52"]):
    d = pd.concat([abl[(abl.тип == "raw") & (abl.модель == m)], abl[(abl.тип == "cum") & (abl.модель == m)]])
    ax[0].errorbar(range(len(d)), d["mean"], yerr=d["std"], marker="o", capsize=3, label=m.upper(), color=col)
ax[0].set_xticks(range(6)); ax[0].set_xticklabels(["сирі"] + [f"+{LABEL[s]}" for s in STEPS], rotation=30, ha="right")
ax[0].set_ylabel("macro-F1 (CV ± σ)"); ax[0].set_title("Кумулятивний приріст метрики"); ax[0].legend(); ax[0].grid(alpha=.3)

# (б) внесок кожного кроку: «лише крок» і «усі крім кроку»
width = 0.2
raw = {m: abl[(abl.тип == "raw") & (abl.модель == m)]["mean"].iloc[0] for m in ABL_MODELS}
full = {m: abl[(abl.тип == "cum") & (abl.крок == "pca") & (abl.модель == m)]["mean"].iloc[0] for m in ABL_MODELS}
x = np.arange(len(STEPS))
for k, (m, col) in enumerate(zip(ABL_MODELS, ["#4c72b0", "#c44e52"])):
    add = [abl[(abl.тип == "add") & (abl.крок == s) & (abl.модель == m)]["mean"].iloc[0] - raw[m] for s in STEPS]
    drp = [full[m] - abl[(abl.тип == "drop") & (abl.крок == s) & (abl.модель == m)]["mean"].iloc[0] for s in STEPS]
    ax[1].bar(x + (2 * k) * width, add, width, label=f"{m.upper()}: Δ від додавання до сирих", color=col, alpha=.9)
    ax[1].bar(x + (2 * k + 1) * width, drp, width, label=f"{m.upper()}: Δ від вилучення з повної", color=col, alpha=.45, hatch="//")
ax[1].axhline(0, color="k", lw=.8); ax[1].set_xticks(x + 1.5 * width); ax[1].set_xticklabels([LABEL[s] for s in STEPS], rotation=20)
ax[1].set_title("Внесок окремих кроків (ΔF1)"); ax[1].legend(fontsize=7); ax[1].grid(alpha=.3)
plt.tight_layout(); savefig("r5_ablation"); plt.show()

In [ ]:
print("Автоматичний чернетковий висновок (перевірте й відредагуйте):\n")
for m in ABL_MODELS:
    gains, noeff = {}, []
    for s in STEPS:
        a = abl[(abl.тип == "add") & (abl.крок == s) & (abl.модель == m)].iloc[0]
        gains[s] = a["mean"] - raw[m]
        if abs(gains[s]) < a["std"]:
            noeff.append(LABEL[s])
    best = max(gains, key=gains.get)
    print(f"[{m.upper()}] сирі: {raw[m]:.4f} → усі кроки: {full[m]:.4f} (Δ = {full[m]-raw[m]:+.4f})")
    print(f"   найбільший ефект при додаванні до сирих: «{LABEL[best]}» ({gains[best]:+.4f})")
    print(f"   ефект у межах σ (нічого не додали): {', '.join(noeff) if noeff else '—'}\n")

### Висновок до результату 5 (напишіть своїми словами за таблицею й графіком)
Шаблон: «Найбільший приріст дали … (Δ = …), особливо для … моделі. Масштабування не вплинуло на RandomForest (дерева інваріантні до масштабу), але суттєво вплинуло на логістичну регресію. PCA зменшила розмірність з … до … ознак і час навчання, але/і не покращила F1. Кроки …, ефект яких менший за σ, не дали достовірного приросту.»

## Артефакти для здачі: README, requirements, декларація ШІ
Наступна комірка створює `README.md`, `requirements.txt` та шаблон `AI_DECLARATION.md`. **Декларацію ШІ потрібно заповнити чесно й самостійно** (інструмент, призначення, запити, як перевірено).

In [ ]:
try:
    import umap as _u; umap_v = _u.__version__
except Exception:
    umap_v = "не встановлено"
readme = f"""# ЛР 5. Очищення даних, інженерія ознак, зменшення розмірності

Задача: класифікація цінового сегмента нерухомості (3 класи), метрика macro-F1, перевірка TimeSeriesSplit(5).

## Версії
Python {platform.python_version()}, scikit-learn {sklearn.__version__}, pandas {pd.__version__}, numpy {np.__version__}, umap-learn {umap_v}

## Дані
`Melbourne_housing_FULL.csv` (Kaggle: Melbourne Housing Market). Покласти поряд із ноутбуком або задати змінну середовища LAB5_DATA.

## Запуск
```
pip install -r requirements.txt
jupyter nbconvert --to notebook --execute lab5.ipynb      # повний прогін
LAB5_QUICK=1 jupyter nbconvert --execute ...              # швидкий прогін (менше дерев/точок)
```
Фіксований seed: {SEED}. Рисунки зберігаються в `figures/`.
"""
reqs = f"numpy\npandas\nscipy\nscikit-learn>={sklearn.__version__.split('.')[0]}.{sklearn.__version__.split('.')[1]}\nmatplotlib\numap-learn\njupyter\n"
decl = """# Декларація використання ШІ (ЗАПОВНІТЬ САМОСТІЙНО)

- **Інструмент:** (назва та версія)
- **Призначення:** (напр. допомога зі структурою ноутбука / налагодження коду)
- **Запити:** (ключові промпти або їх короткий зміст)
- **Як перевірено:** (що запускали, що змінили, які частини розумію й можу пояснити)
"""
Path("README.md").write_text(readme, encoding="utf-8")
Path("requirements.txt").write_text(reqs)
if not Path("AI_DECLARATION.md").exists():
    Path("AI_DECLARATION.md").write_text(decl, encoding="utf-8")
print("Створено: README.md, requirements.txt, AI_DECLARATION.md")